<a href="https://colab.research.google.com/github/hvs515/Federated-Learning-IID-simulation/blob/main/FedIID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import copy
import random
from torch.utils.data import DataLoader, Subset
from torch import nn, optim

BATCH_SIZE = 32
SEED = 42
TOTAL_CLIENTS = 100
CLIENTS_PER_ROUND = 10
COMMUNICATION_ROUNDS = 30
LOCAL_EPOCHS = 5
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_CLASSES = 10

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [8]:
def create_iid_clients(train_dataset, num_clients, batch_size, seed):
    np.random.seed(seed)

    num_samples = len(train_dataset)
    samples_per_client = num_samples // num_clients

    all_indices = list(range(num_samples))
    np.random.shuffle(all_indices)

    client_dataloaders = []
    for i in range(num_clients):
        start_idx = i * samples_per_client
        end_idx = (i + 1) * samples_per_client
        client_indices = all_indices[start_idx:end_idx]

        client_dataset = Subset(train_dataset, client_indices)
        client_dataloader = DataLoader(client_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        client_dataloaders.append(client_dataloader)

    return client_dataloaders

cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2471, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

num_samples = len(train_dataset)
samples_per_client = num_samples // TOTAL_CLIENTS

client_dataloaders = create_iid_clients(train_dataset, TOTAL_CLIENTS, BATCH_SIZE, SEED)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\nVerifying IID client data distribution for 3 random clients:")
random_client_ids = random.sample(range(TOTAL_CLIENTS), 3)

for client_id in random_client_ids:
    print(f"\nClient {client_id} Class Distribution:")
    dataloader = client_dataloaders[client_id]
    class_counts = {i: 0 for i in range(NUM_CLASSES)}
    total_samples = 0

    for _, labels in dataloader:
        for label in labels:
            class_counts[label.item()] += 1
        total_samples += len(labels)

    for class_id, count in class_counts.items():
        percentage = (count / total_samples) * 100 if total_samples > 0 else 0
        print(f"  Class {class_id}: {count} samples ({percentage:.2f}%) - {(count / samples_per_client) * 100 if samples_per_client > 0 else 0:.2f}% of client's total samples")
    print(f"  Total samples for client {client_id}: {total_samples}")

100%|██████████| 170M/170M [00:02<00:00, 64.6MB/s]



Verifying IID client data distribution for 3 random clients:

Client 81 Class Distribution:
  Class 0: 52 samples (10.83%) - 10.40% of client's total samples
  Class 1: 42 samples (8.75%) - 8.40% of client's total samples
  Class 2: 41 samples (8.54%) - 8.20% of client's total samples
  Class 3: 47 samples (9.79%) - 9.40% of client's total samples
  Class 4: 47 samples (9.79%) - 9.40% of client's total samples
  Class 5: 39 samples (8.12%) - 7.80% of client's total samples
  Class 6: 58 samples (12.08%) - 11.60% of client's total samples
  Class 7: 46 samples (9.58%) - 9.20% of client's total samples
  Class 8: 56 samples (11.67%) - 11.20% of client's total samples
  Class 9: 52 samples (10.83%) - 10.40% of client's total samples
  Total samples for client 81: 480

Client 14 Class Distribution:
  Class 0: 53 samples (11.04%) - 10.60% of client's total samples
  Class 1: 41 samples (8.54%) - 8.20% of client's total samples
  Class 2: 54 samples (11.25%) - 10.80% of client's total sampl

In [9]:
import torchvision.models as models
from torch import nn
import torch

NUM_CLASSES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} (defined in model cell for robustness)")

def get_model():
    model = models.resnet18(pretrained=False)

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

    model = model.to(device)
    return model

def get_model_weights(model):
    return model.state_dict()

def set_model_weights(model, weights):
    model.load_state_dict(weights)

global_model = get_model()
print(f"Global model (ResNet-18) initialized and moved to {device}. Number of classes in final layer: {global_model.fc.out_features}")
print("Model architecture snippet:")
print(global_model)

Using device: cpu (defined in model cell for robustness)
Global model (ResNet-18) initialized and moved to cpu. Number of classes in final layer: 10
Model architecture snippet:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [10]:
def local_train(model, dataloader, epochs, lr):
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM)

    total_samples = 0

    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_samples += len(data)

    return model.state_dict(), total_samples

print("local_train function defined successfully.")

local_train function defined successfully.


In [11]:
def fedavg(global_model, client_weights_list, client_sample_counts):
    if not client_weights_list:
        return global_model

    total_samples = sum(client_sample_counts)

    aggregated_weights = client_weights_list[0].copy()
    for key in aggregated_weights.keys():
        aggregated_weights[key] = torch.zeros_like(aggregated_weights[key], dtype=torch.float)

    for i, client_weights in enumerate(client_weights_list):
        num_samples = client_sample_counts[i]
        weight_factor = num_samples / total_samples

        for key in aggregated_weights.keys():
            aggregated_weights[key] += client_weights[key].float() * weight_factor

    for key in global_model.state_dict().keys():
        if global_model.state_dict()[key].dtype == torch.long:
            aggregated_weights[key] = aggregated_weights[key].round().long()

    global_model.load_state_dict(aggregated_weights)
    return global_model

print("fedavg function defined successfully.")

fedavg function defined successfully.


In [ ]:
import random
import copy

TOTAL_CLIENTS = 100
CLIENTS_PER_ROUND = 10
COMMUNICATION_ROUNDS = 30
LOCAL_EPOCHS = 5
LEARNING_RATE = 0.01

def evaluate_model(model, dataloader):
    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    criterion = nn.CrossEntropyLoss(reduction='sum')

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total_samples += len(data)

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples
    return avg_loss, accuracy

global_model = get_model()

accuracies = []
losses = []

print("\nFederated Training Loop Started")
print("-----------------------------------------------------------------------------------------------------------------")
print(f"| {'Round':<5} | {'Test Accuracy':<15} | {'Test Loss':<12} | {'Selected Clients':<20} | {'Total Samples Processed':<25} |")
print("-----------------------------------------------------------------------------------------------------------------")

for round_num in range(1, COMMUNICATION_ROUNDS + 1):
    selected_client_indices = random.sample(range(TOTAL_CLIENTS), CLIENTS_PER_ROUND)

    client_weights_list = []
    client_sample_counts = []
    total_samples_processed_round = 0

    for client_id in selected_client_indices:
        local_model = copy.deepcopy(global_model)

        trained_weights, num_samples = local_train(
            local_model,
            client_dataloaders[client_id],
            LOCAL_EPOCHS,
            LEARNING_RATE
        )

        client_weights_list.append(trained_weights)
        client_sample_counts.append(num_samples)
        total_samples_processed_round += num_samples

    global_model = fedavg(global_model, client_weights_list, client_sample_counts)

    test_loss, test_accuracy = evaluate_model(global_model, test_dataloader)
    accuracies.append(test_accuracy)
    losses.append(test_loss)

    print(f"| {round_num:<5} | {test_accuracy:.4f}{' ':>11} | {test_loss:.4f}{' ':>7} | {str(selected_client_indices):<20} | {total_samples_processed_round:<25} |")

print("-----------------------------------------------------------------------------------------------------------------")
print("Federated Training Loop Finished.")


Federated Training Loop Started
-----------------------------------------------------------------------------------------------------------------
| Round | Test Accuracy   | Test Loss    | Selected Clients     | Total Samples Processed   |
-----------------------------------------------------------------------------------------------------------------
| 1     | 0.1000            | 2.3236        | [94, 35, 31, 28, 17, 13, 86, 69, 11, 75] | 24000                     |
| 2     | 0.1165            | 2.2459        | [54, 4, 3, 11, 27, 29, 64, 77, 71, 25] | 24000                     |
| 3     | 0.2903            | 1.8936        | [91, 83, 89, 69, 53, 28, 57, 75, 35, 0] | 24000                     |
| 4     | 0.3834            | 1.6685        | [97, 20, 89, 54, 43, 35, 19, 27, 13, 11] | 24000                     |
| 5     | 0.4378            | 1.5344        | [48, 12, 45, 44, 77, 33, 5, 93, 58, 68] | 24000                     |
| 6     | 0.4750            | 1.4408        | [15, 48, 10, 70, 3